1. Introduction

2. Data Acquisition

In [1]:
import requests
import pandas as pd
import json
import time
import os

In [ ]:
with open("../keys.json") as f:
    GUARDIAN_KEY = json.load(f)["guardian"]["api_key"]

In [3]:
COUNTRIES = [
    "ukraine", "sudan", "syria", "yemen", "ethiopia", "afghanistan",
    "iraq", "somalia", "democratic-republic-of-congo", "myanmar",
    "nigeria", "mali", "mozambique", "haiti", "libya",
    "central-african-republic", "south-sudan", "burkina-faso",
    "cameroon", "israel", "pakistan", "russia", "united-states",
    "france", "germany", "spain", "italy", "greece"
]

In [4]:
FROM_DATE = "2015-01-01"
TO_DATE   = "2024-12-31"

BASE_URL = "https://content.guardianapis.com/search"

def fetch_articles(country_slug, from_date, to_date, api_key):
    """
    Fetches all Guardian articles tagged with a given country
    within the specified date range. Paginates automatically.

    Returns a list of article dictionaries.
    """
    articles = []
    page = 1

    while True:
        params = {
            "tag":          f"world/{country_slug}",
            "from-date":    from_date,
            "to-date":      to_date,
            "show-fields":  "wordcount,sectionName,bodyText",
            "page-size":    200,
            "page":         page,
            "api-key":      api_key
        }

        response = requests.get(BASE_URL, params=params)

        # Respect rate limits — free tier allows 12 requests/second
        time.sleep(0.1)

        if response.status_code != 200:
            print(f"  Error {response.status_code} for {country_slug} page {page}")
            break

        data = response.json()["response"]
        results = data.get("results", [])

        if not results:
            break

        for article in results:
            articles.append({
                "country":      country_slug,
                "date":         article["webPublicationDate"],
                "section":      article["sectionName"],
                "wordcount":    article.get("fields", {}).get("wordcount", None),
                "title":        article["webTitle"],
                "pillar":       article.get("pillarName", None)
            })

        # Stop if we have reached the last page
        if page >= data["pages"]:
            break

        page += 1

    return articles

In [5]:
RAW_PATH = "../data/guardian_raw.csv"

if os.path.exists(RAW_PATH):
    # If raw data already collected, load from file — do not re-query the API
    print("Raw Guardian data already exists — loading from file.")
    guardian_raw = pd.read_csv(RAW_PATH)
else:
    all_articles = []
    for country in COUNTRIES:
        print(f"Fetching: {country}...")
        articles = fetch_articles(country, FROM_DATE, TO_DATE, GUARDIAN_KEY)
        all_articles.extend(articles)
        print(f"  -> {len(articles)} articles collected")

    guardian_raw = pd.DataFrame(all_articles)
    guardian_raw.to_csv(RAW_PATH, index=False)
    print(f"\nSaved {len(guardian_raw)} articles to {RAW_PATH}")

print(f"\nShape: {guardian_raw.shape}")
print(f"Date range: {guardian_raw['date'].min()} to {guardian_raw['date'].max()}")
guardian_raw.head()

Fetching: ukraine...
  Error 401 for ukraine page 1
  -> 0 articles collected
Fetching: sudan...
  Error 401 for sudan page 1
  -> 0 articles collected
Fetching: syria...
  Error 401 for syria page 1
  -> 0 articles collected
Fetching: yemen...
  Error 401 for yemen page 1
  -> 0 articles collected
Fetching: ethiopia...
  Error 401 for ethiopia page 1
  -> 0 articles collected
Fetching: afghanistan...
  Error 401 for afghanistan page 1
  -> 0 articles collected
Fetching: iraq...
  Error 401 for iraq page 1
  -> 0 articles collected
Fetching: somalia...
  Error 401 for somalia page 1
  -> 0 articles collected
Fetching: democratic-republic-of-congo...
  Error 401 for democratic-republic-of-congo page 1
  -> 0 articles collected
Fetching: myanmar...
  Error 401 for myanmar page 1
  -> 0 articles collected
Fetching: nigeria...
  Error 401 for nigeria page 1
  -> 0 articles collected
Fetching: mali...
  Error 401 for mali page 1
  -> 0 articles collected
Fetching: mozambique...
  Error 401 

KeyError: 'date'

In [6]:
ACLED_PATH = "../data/acled_fatalities.csv"

acled_raw = pd.read_csv(ACLED_PATH)

print(f"Shape: {acled_raw.shape}")
print(f"Columns: {list(acled_raw.columns)}")
print(f"Countries: {acled_raw['country'].nunique()} unique countries")
print(f"Year range: {acled_raw['year'].min()} to {acled_raw['year'].max()}")
acled_raw.head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/acled_fatalities.csv'

In [ ]:
OWID_URL = (
    "https://ourworldindata.org/grapher/world-bank-income-groups.csv"
    "?v=1&csvType=full&useColumnShortNames=false"
)

wb_raw = pd.read_csv(
    OWID_URL,
    storage_options={"User-Agent": "Our World In Data data fetch/1.0"}
)

print(f"Shape: {wb_raw.shape}")
print(f"Columns: {list(wb_raw.columns)}")
print(f"Years available: {wb_raw['Year'].min()} to {wb_raw['Year'].max()}")
wb_raw.head()

# We filter to the most recent available year to get a single
# classification per country for the purpose of our binary label.
wb_latest = wb_raw[wb_raw["Year"] == wb_raw["Year"].max()].copy()
wb_latest.to_csv("../data/worldbank_income.csv", index=False)
print(f"\nSaved World Bank classification to data/worldbank_income.csv")
print(f"Countries classified: {wb_latest['Entity'].nunique()}")

3. Preparation of the Data for Analysis

4. Exploratory Data Analysis

5. Conclusion